Customer Data Profiling

In [ ]:
SELECT
   COUNT(*) AS total_rows,
   COUNT(DISTINCT customer_id) AS unique_customers
FROM data_analysis.customer_raw_data ;

Find Duplicate Customers

In [ ]:
SELECT
   customer_id
   COUNT (*) AS occurance_count
FROM data_analysis.customer_raw_data
GROUP BY customer_id
HAVING COUNT(*)>1
ORDER BY occurance_count DESC;

Orders Data Profiling

In [ ]:
SELECT
   COUNT (*) AS total_rows
   COUNT(DISTINCT order_id) AS unique_orders
FROM data_analysis.orders_raw_data;

Find Duplicate Orders

In [ ]:
SELECT
    order_id,
  COUNT(*) AS occurance_count
FROM data_analysis.orders_raw_data
GROUP BY order_id
HAVING COUNT(*)>1
ORDER BY occurance_count;

Missing Customer IDs

In [ ]:
SELECT
    COUNT (*) AS null_customer_IDS
FROM data_analysis.customer_raw_data
WHERE customer_id IS NULL;

Missing Product IDs

In [ ]:
SELECT
 COUNT(*) AS null_product_IDS
FROM data_analysis.products_raw_data
WHERE product_id IS NULL;

Invaild Order Dates

In [ ]:
SELECT
  COUNT(*) AS Invalid_dates
FROM data_analysis.orders_raw_data
WHERE SAFE.PARSE_DATETIME('%Y-%m-%d %H:%M:%S' , TRIM(order_date)) IS NULL

Negative Quantities/Returns

In [ ]:
SELECT
  COUNT(*) AS negative_sales
FROM data_analysis.orders_raw_data
WHERE sales_amount < 0 ;

Negative Profit

In [ ]:
SELECT
  COUNT(*) AS negative_profit
FROM data_analysis.orders_raw_data
WHERE profit < 0 ;

-- Did not remove any negatvie profit because it represents legitimate loss-making transactions --

Discount Validation

In [ ]:
SELECT
 MIN(discount) AS minimum_discount ,
 MAX (discount) AS maximum_discount
FROM data_analysis.orders_raw_data

-- Discount were within expected range --

Produt category Check

In [ ]:
SELECT
  category
FROM data_analysis.products_raw_data
ORDER BY category;

---- Staging Layer----

Clean Customers

In [ ]:
CREATE OR REPLACE TABLE data_analysis.stg_customers
AS
SELECT DISTINCT
  TRIM(customer_id) AS customer_id,
  TRIM(customer_name)AS customer_name,
  TRIM(gender) AS gender
  age,
  TRIM(customer_segment) AS customer_segment,
  SAFE_CAST(signup_date AS DATE) AS signup_date
FROM data_analysis.customers_raw_data;

Clean Products

In [ ]:
CREATE OR REPLACE data_analysis.stg_products
AS
SELECT DISTINCT
  TRIM(product_id) AS product_id
  TRIM(product_name) AS product_name
  INITCAP(TRIM(category)) AS category
  INITCAP(TRIM(sub_category)) AS sub_category
  cost,
  selling_price
FROM data_analysis.products_raw_data;

Clean Regions

In [ ]:
CREATE OR REPLACE TABLE data_analysis.stg_regions
AS
SELECT
  TRIM(string_field_0) AS region_id
  TRIM(string_field_1) AS state,
  TRIM(string_field_2) AS city,
  TRIM(string_field_3) AS region
  FROM
  data_analysis.regions_raw_data
  WHERE LOWER(TRIM(string_field_0)) != 'region_id';

Clean Orders

In [ ]:
CREATE OR REPLACE TABLE data_analysis.stg_orders
AS
SELECT DISTINCT
  TRIM(order_id) AS order_id,
  SAFE.PARSE_DATETIME('%Y-%m-%d %H:%M:%S', TRIM(order_date)) AS order_datetime,
  TRIM(customer_id) AS customer_id,
  TRIM(product_id) AS product_id,
  TRIM(region_id) AS region_id,
  quantity,
  discount,
  sales_amount,
  cost_amount,
  profit,
  CASE
      WHEN quantity < 0 THEN
  'RETURN'
          ELSE 'SALE'
     END AS transaction_type
  FROM data_analysis.orders_raw_data;

Staging Validation

In [ ]:
---Orders---
SELECT
  COUNT(*) AS total_rows,
  COUNT(DISTINCT order_id) AS unique_orders
FROM data_analysis.stg_orders;

In [ ]:
---Returns---
SELECT
   transaction_type,
   COUNT(*) AS transaction_count
FROM data_analysis.stg_orders
GROUP BY transaction_type;

Orphan Record Checks

In [ ]:
---Customers----
SELECT COUNT(*) AS orphan_customers
FROM `project-5c2790f9-0cd9-4002-90e.data_analysis.stg_orders` o
LEFT JOIN `project-5c2790f9-0cd9-4002-90e.data_analysis.stg_customers` c
    ON o.customer_id = c.customer_id
WHERE o.customer_id IS NOT NULL
  AND c.customer_id IS NULL;

In [ ]:
---Products---
SELECT COUNT(*) AS orphan_products
FROM `project-5c2790f9-0cd9-4002-90e.data_analysis.stg_orders` o
LEFT JOIN `project-5c2790f9-0cd9-4002-90e.data_analysis.stg_products` p
    ON o.product_id = p.product_id
WHERE o.product_id IS NOT NULL
  AND p.product_id IS NULL;

In [ ]:
---Regions---
SELECT COUNT(*) AS orphan_regions
FROM `project-5c2790f9-0cd9-4002-90e.data_analysis.stg_orders` o
LEFT JOIN `project-5c2790f9-0cd9-4002-90e.data_analysis.stg_regions` r
    ON o.region_id = r.region_id
WHERE o.region_id IS NOT NULL
  AND r.region_id IS NULL;

Customer Dimension

In [ ]:
CREATE OR REPLACE TABLE
`project-5c2790f9-0cd9-4002-90e.data_analysis.dim_customer`
AS

SELECT
    ROW_NUMBER() OVER (ORDER BY customer_id) AS customer_key,
    customer_id,
    customer_name,
    gender,
    age,
    customer_segment,
    signup_date
FROM
`project-5c2790f9-0cd9-4002-90e.data_analysis.stg_customers`;

In [ ]:
--- Then the Unknown Customer member was added:---
INSERT INTO
`project-5c2790f9-0cd9-4002-90e.data_analysis.dim_customer`
(
    customer_key,
    customer_id,
    customer_name,
    gender,
    age,
    customer_segment,
    signup_date
)

VALUES
(
    0,
    'UNKNOWN',
    'Unknown Customer',
    'Unknown',
    NULL,
    'Unknown',
    NULL
);

Product Dimension

In [ ]:
CREATE OR REPLACE TABLE
`project-5c2790f9-0cd9-4002-90e.data_analysis.dim_product`
AS

SELECT
    ROW_NUMBER() OVER (ORDER BY product_id) AS product_key,
    product_id,
    product_name,
    category,
    sub_category,
    cost,
    selling_price
FROM
`project-5c2790f9-0cd9-4002-90e.data_analysis.stg_products`;

In [ ]:
---Unknown Product---
INSERT INTO
`project-5c2790f9-0cd9-4002-90e.data_analysis.dim_product`
(
    product_key,
    product_id,
    product_name,
    category,
    sub_category,
    cost,
    selling_price
)

VALUES
(
    0,
    'UNKNOWN',
    'Unknown Product',
    'Unknown',
    'Unknown',
    NULL,
    NULL
);

Region Dimensions

In [ ]:
CREATE OR REPLACE TABLE
`project-5c2790f9-0cd9-4002-90e.data_analysis.dim_region`
AS

SELECT
    ROW_NUMBER() OVER (ORDER BY region_id) AS region_key,
    region_id,
    state,
    city,
    region
FROM
`project-5c2790f9-0cd9-4002-90e.data_analysis.stg_regions`;

Date Dimension

In [ ]:
CREATE OR REPLACE TABLE
`project-5c2790f9-0cd9-4002-90e.data_analysis.dim_date`
AS

SELECT
    CAST(FORMAT_DATE('%Y%m%d', full_date) AS INT64) AS date_key,
    full_date,
    EXTRACT(YEAR FROM full_date) AS year,
    EXTRACT(QUARTER FROM full_date) AS quarter,
    EXTRACT(MONTH FROM full_date) AS month,
    FORMAT_DATE('%B', full_date) AS month_name,
    EXTRACT(WEEK FROM full_date) AS week,
    EXTRACT(DAY FROM full_date) AS day,
    FORMAT_DATE('%A', full_date) AS day_name
FROM UNNEST(
    GENERATE_DATE_ARRAY(
        '2024-01-01',
        '2025-12-31'
    )
) AS full_date;

FACT TABLE

In [ ]:
CREATE OR REPLACE TABLE
`project-5c2790f9-0cd9-4002-90e.data_analysis.fact_orders`
AS

SELECT
    ROW_NUMBER() OVER (ORDER BY o.order_id) AS order_key,

    o.order_id,

    COALESCE(c.customer_key, 0) AS customer_key,
    COALESCE(p.product_key, 0) AS product_key,
    COALESCE(r.region_key, 0) AS region_key,

    CASE
        WHEN o.order_datetime IS NULL THEN NULL
        ELSE CAST(
            FORMAT_DATE(
                '%Y%m%d',
                DATE(o.order_datetime)
            ) AS INT64
        )
    END AS date_key,

    o.quantity,
    o.discount,
    o.sales_amount,
    o.cost_amount,
    o.profit,
    o.transaction_type

FROM
`project-5c2790f9-0cd9-4002-90e.data_analysis.stg_orders` o

LEFT JOIN
`project-5c2790f9-0cd9-4002-90e.data_analysis.dim_customer` c
ON o.customer_id = c.customer_id

LEFT JOIN
`project-5c2790f9-0cd9-4002-90e.data_analysis.dim_product` p
ON o.product_id = p.product_id

LEFT JOIN
`project-5c2790f9-0cd9-4002-90e.data_analysis.dim_region` r
ON o.region_id = r.region_id;

Final Validation

In [ ]:
SELECT
    COUNT(*) AS total_rows,
    COUNT(DISTINCT order_id) AS unique_orders,

    COUNTIF(customer_key = 0) AS unknown_customers,
    COUNTIF(product_key = 0) AS unknown_products,
    COUNTIF(region_key = 0) AS unknown_regions,
    COUNTIF(date_key IS NULL) AS invalid_dates

FROM
`project-5c2790f9-0cd9-4002-90e.data_analysis.fact_orders`;